# 📚 Unidad 1: Material Complementario - Teoría
## Módulo 04 - Estadística Descriptiva Avanzada
### Laboratorio - Universidad del Aconcagua

---

## 🎯 Objetivos

1. ✅ Dominar métricas avanzadas (asimetría, curtosis, percentiles)
2. ✅ Aplicar estadística robusta (mediana, MAD, percentiles)
3. ✅ Realizar análisis de distribuciones (KS test, Shapiro-Wilk)
4. ✅ Implementar bootstrapping para intervalos de confianza

---

## 1️⃣ Métricas de Forma de Distribución

### 📊 Asimetría (Skewness)

**Mide la simetría** de la distribución:

* **Skew = 0**: Distribución **simétrica** (normal)
* **Skew > 0**: **Asimetría positiva** (cola derecha larga)
* **Skew < 0**: **Asimetría negativa** (cola izquierda larga)

```python
from scipy.stats import skew
skewness = skew(df['columna'])

if abs(skewness) < 0.5:
    print("✅ Distribución simétrica")
elif skewness > 0.5:
    print("⚠️ Asimetría positiva (cola derecha)")
else:
    print("⚠️ Asimetría negativa (cola izquierda)")
```

**Impacto:**
* Media **sesgada** hacia la cola
* Usar **mediana** en lugar de media
* Considerar **transformación** (log, Box-Cox)

---

### 📉 Curtosis (Kurtosis)

**Mide el "peso" de las colas**:

* **Kurt = 3**: Normal (mesocúrtica)
* **Kurt > 3**: **Leptocúrtica** (colas pesadas, más outliers)
* **Kurt < 3**: **Platocúrtica** (colas ligeras, pocos outliers)

```python
from scipy.stats import kurtosis
kurt = kurtosis(df['columna'], fisher=True)  # fisher=True para excess kurtosis (Kurt-3)

if abs(kurt) < 0.5:
    print("✅ Similar a normal")
elif kurt > 0.5:
    print("⚠️ Colas pesadas (más outliers que normal)")
else:
    print("📊 Colas ligeras (pocos outliers)")
```

---

## 2️⃣ Estadística Robusta

### ¿Por qué Robusta?

**Problema**: Media y desviación estándar son **muy sensibles a outliers**.

| Métrica Clásica | Métrica Robusta | Ventaja |
|------------------|-------------------|----------|
| **Media** | **Mediana** | No afectada por outliers |
| **Desv. Estándar** | **MAD** (Median Absolute Deviation) | Robusta a extremos |
| **Rango** | **IQR** (Rango Intercuartílico) | Ignora extremos |

### 🛡️ MAD (Median Absolute Deviation)

```python
def mad(data):
    median = np.median(data)
    return np.median(np.abs(data - median))

mad_value = mad(df['columna'])
print(f"MAD: {mad_value:.2f}")

# Detectar outliers con MAD
median = df['columna'].median()
mod_z_score = 0.6745 * (df['columna'] - median) / mad_value
outliers = df[np.abs(mod_z_score) > 3.5]
print(f"Outliers detectados (MAD): {len(outliers)}")
```

---

## 3️⃣ Tests de Normalidad

### 📋 ¿Por qué Testear Normalidad?

Muchos tests estadísticos **asumen distribución normal**:
* t-test
* ANOVA
* Regresión lineal (residuos)

### Test de Shapiro-Wilk (datasets pequeños, n < 5000)

```python
from scipy.stats import shapiro
stat, p_value = shapiro(df['columna'])

if p_value > 0.05:
    print(f"✅ Normal (p={p_value:.4f})")
else:
    print(f"❌ No normal (p={p_value:.4f})")
```

### Test de Kolmogorov-Smirnov (datasets grandes)

```python
from scipy.stats import kstest
stat, p_value = kstest(df['columna'], 'norm', args=(df['columna'].mean(), df['columna'].std()))

if p_value > 0.05:
    print(f"✅ Normal (p={p_value:.4f})")
else:
    print(f"❌ No normal (p={p_value:.4f})")
```

**💡 Si NO es normal:**
* Usar **tests no paramétricos** (Mann-Whitney, Kruskal-Wallis)
* **Transformar** datos (log, Box-Cox)
* Usar **estadística robusta** (mediana, MAD)

---

## 4️⃣ Bootstrapping

**🎯 Objetivo**: Estimar **intervalo de confianza** sin asumir distribución.

**🔄 Cómo funciona:**
1. Tomar **muestra con reemplazo** del dataset original
2. Calcular estadística (media, mediana, etc.)
3. Repetir 1000+ veces
4. Obtener distribución de la estadística

```python
def bootstrap_ci(data, stat_func=np.mean, n_bootstrap=1000, confidence=0.95):
    """
    Calcula intervalo de confianza por bootstrapping
    """
    bootstrap_stats = []
    
    for _ in range(n_bootstrap):
        # Muestra con reemplazo
        sample = np.random.choice(data, size=len(data), replace=True)
        bootstrap_stats.append(stat_func(sample))
    
    # Percentiles para IC
    alpha = (1 - confidence) / 2
    lower = np.percentile(bootstrap_stats, alpha * 100)
    upper = np.percentile(bootstrap_stats, (1 - alpha) * 100)
    
    return lower, upper, bootstrap_stats

# Ejemplo
lower, upper, dist = bootstrap_ci(df['monto'], stat_func=np.median, confidence=0.95)
print(f"IC 95% para mediana: [{lower:.2f}, {upper:.2f}]")
```

**✅ Ventajas:**
* **No asume distribución** (no paramétrico)
* Funciona con **cualquier estadística** (media, mediana, varianza, percentiles)
* Útil para **datasets pequeños**

---

## 5️⃣ Percentiles y Cuartiles

**Percentiles** dividen datos en 100 partes iguales:

```python
# Percentiles clave
p25 = df['monto'].quantile(0.25)  # Q1
p50 = df['monto'].quantile(0.50)  # Mediana (Q2)
p75 = df['monto'].quantile(0.75)  # Q3
p90 = df['monto'].quantile(0.90)  # P90
p95 = df['monto'].quantile(0.95)  # P95
p99 = df['monto'].quantile(0.99)  # P99

print(f"""Percentiles de monto:
  P25 (Q1): ${p25:.2f}
  P50 (Mediana): ${p50:.2f}
  P75 (Q3): ${p75:.2f}
  P90: ${p90:.2f} (Top 10%)
  P95: ${p95:.2f} (Top 5%)
  P99: ${p99:.2f} (Top 1%)
""")
```

**📊 Aplicaciones:**
* **Segmentación**: Clientes por quintiles de ingreso
* **Benchmarking**: "Estás en el percentil 75 de ventas"
* **SLAs**: "99% de requests < 200ms" (P99)

---

## ✅ Resumen

### 📊 Cuándo Usar Cada Métrica

| Situación | Métrica Recomendada |
|-----------|------------------------|
| **Distribución simétrica, sin outliers** | Media, Desv. Estándar |
| **Outliers presentes** | Mediana, MAD, IQR |
| **Distribución sesgada** | Mediana, Percentiles |
| **Necesitas IC sin asumir normalidad** | Bootstrapping |
| **Caracterizar forma** | Skewness, Kurtosis |
| **Validar normalidad** | Shapiro-Wilk, KS test |

### 💡 Mejores Prácticas

* ✅ **Siempre** visualiza distribuciones antes de calcular estadísticas
* ✅ Usa **estadística robusta** por defecto (mediana > media)
* ✅ Reporta **percentiles clave** (P25, P50, P75, P90, P95)
* ✅ **Testea normalidad** antes de usar tests paramétricos
* ✅ Aplica **bootstrapping** cuando hay duda sobre la distribución

---

### 🚀 Próximos Pasos

* Practica con ejercicios del Módulo 04
* Integra con Módulos anteriores (EDA, missing, correlaciones)
* Aplica en tus Trabajos Prácticos

---

**Universidad del Aconcagua - Mendoza, Argentina 🇦🇷**